In [1]:
# Install required packages (Kaggle environment already has most)

import os, sys, json, time, gc
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter, maximum_filter, gaussian_filter1d
from scipy.optimize import linear_sum_assignment
from scipy.spatial import KDTree
from dataclasses import dataclass, field
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

print("Setup complete")

Setup complete


In [2]:
# Physical voxel scale (z, y, x) in micrometres per voxel
SCALE = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)

@dataclass
class ImageVolume:
    path: str
    shape: tuple
    dtype: np.dtype
    chunk: tuple
    
    @property
    def n_t(self) -> int:
        return int(self.shape[0])
    
    def frame(self, t: int) -> np.ndarray:
        return _read_chunk(self.path, t, self.shape, self.dtype)

def open_image(zarr_path: str) -> ImageVolume:
    with open(os.path.join(zarr_path, "0", "zarr.json")) as f:
        meta = json.load(f)
    shape = tuple(int(s) for s in meta["shape"])
    dtype = np.dtype(meta["data_type"])
    return ImageVolume(path=zarr_path, shape=shape, dtype=dtype, chunk=None)

def _read_chunk(zarr_path: str, t: int, shape: tuple, dtype: np.dtype) -> np.ndarray:
    """Read and decode one timepoint chunk -> (Z, Y, X)."""
    frame_shape = shape[1:]
    chunk_path = os.path.join(zarr_path, "0", "c", str(t), "0", "0", "0")
    
    try:
        import blosc2
        with open(chunk_path, "rb") as f:
            raw = f.read()
        dec = blosc2.decompress(raw)
        arr = np.frombuffer(dec, dtype=dtype)
        if arr.size == int(np.prod(frame_shape)):
            return arr.reshape(frame_shape).copy()
    except:
        import zarr
        z = zarr.open(os.path.join(zarr_path, "0"), mode="r")
        return np.asarray(z[t])

@dataclass
class TrackGraph:
    node_t: np.ndarray
    node_z: np.ndarray
    node_y: np.ndarray
    node_x: np.ndarray
    node_ids: np.ndarray
    edges: np.ndarray
    meta: dict
    
    @property
    def n_nodes(self) -> int:
        return len(self.node_ids)
    
    @property
    def n_edges(self) -> int:
        return len(self.edges)
    
    def coords_by_id(self) -> dict:
        out = {}
        for i, nid in enumerate(self.node_ids):
            out[int(nid)] = (int(self.node_t[i]), float(self.node_z[i]),
                           float(self.node_y[i]), float(self.node_x[i]))
        return out

def _ball_footprint(radius_um: float, eff_spacing: np.ndarray) -> np.ndarray:
    rad_vox = np.maximum(1, np.round(radius_um / eff_spacing).astype(int))
    zz, yy, xx = np.ogrid[-rad_vox[0]:rad_vox[0]+1,
                          -rad_vox[1]:rad_vox[1]+1,
                          -rad_vox[2]:rad_vox[2]+1]
    d = ((zz * eff_spacing[0])**2 + (yy * eff_spacing[1])**2 + (xx * eff_spacing[2])**2)
    return d <= radius_um**2

print("Data I/O loaded")

Data I/O loaded


In [3]:
def detect_blobs_enhanced(vol: np.ndarray,
                         xy_downsample: int = 2,
                         min_distance_um: float = 2.5,
                         rel_threshold: float = 0.025,
                         abs_percentile: float = 40.0,
                         max_peaks: int = 60000) -> np.ndarray:
    """Enhanced blob detector with multi-scale detection."""
    
    vf = vol.astype(np.float32)
    ds = vf[:, ::xy_downsample, ::xy_downsample]
    
    eff = np.array([SCALE[0], SCALE[1]*xy_downsample, SCALE[2]*xy_downsample])
    
    # Normalize
    lo, hi = np.percentile(ds, [1.0, 99.5])
    if hi <= lo:
        hi = lo + 1.0
    norm = np.clip((ds - lo) / (hi - lo), 0, None)
    
    # Multi-scale detection
    scales = [[1.2, 3.5], [1.8, 5.0], [2.5, 6.5]]
    all_coords = []
    all_scores = []
    
    for small_um, large_um in scales:
        s_small = small_um / eff
        s_large = large_um / eff
        
        g1 = gaussian_filter(norm, sigma=s_small)
        g2 = gaussian_filter(norm, sigma=s_large)
        dog = g1 - g2
        
        footprint = _ball_footprint(min_distance_um, eff)
        mx = maximum_filter(dog, footprint=footprint, mode="nearest")
        
        thr = max(rel_threshold, np.percentile(dog[dog>0], 50) if np.any(dog>0) else 0)
        abs_thr = np.percentile(norm, abs_percentile)
        
        peaks = (dog == mx) & (dog >= thr) & (norm >= abs_thr)
        coords = np.argwhere(peaks)
        
        if len(coords) > 0:
            vals = dog[peaks]
            if len(coords) > max_peaks // len(scales):
                idx = np.argsort(vals)[-max_peaks // len(scales):]
                coords = coords[idx]
                vals = vals[idx]
            
            coords = coords.astype(np.float64)
            coords[:, 1] *= xy_downsample
            coords[:, 2] *= xy_downsample
            
            all_coords.append(coords)
            all_scores.append(vals)
    
    if not all_coords:
        return np.zeros((0, 3), dtype=np.float64)
    
    # Merge peaks
    all_coords = np.vstack(all_coords)
    all_scores = np.concatenate(all_scores)
    
    # Sort by score
    idx = np.argsort(all_scores)[::-1]
    all_coords = all_coords[idx]
    all_scores = all_scores[idx]
    
    # Non-maximum suppression
    keep = []
    for i, coord in enumerate(all_coords):
        if len(keep) == 0:
            keep.append(i)
        else:
            distances = np.sqrt(((all_coords[keep] - coord) * SCALE)**2)
            distances = np.sqrt((distances**2).sum(axis=1))
            if np.min(distances) >= min_distance_um:
                keep.append(i)
        if len(keep) >= max_peaks:
            break
    
    return all_coords[keep]

def refine_centroids(vol: np.ndarray, coords: np.ndarray, win=(1, 3, 3)) -> np.ndarray:
    """Intensity-weighted center of mass refinement."""
    if len(coords) == 0:
        return coords
    
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64)
    wz, wy, wx = win
    
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1)
        y0, y1 = max(0, y-wy), min(Y, y+wy+1)
        x0, x1 = max(0, x-wx), min(X, x+wx+1)
        
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64)
        s = patch.sum()
        if s <= 0:
            continue
        
        zz = np.arange(z0, z1)[:, None, None]
        yy = np.arange(y0, y1)[None, :, None]
        xx = np.arange(x0, x1)[None, None, :]
        
        out[i, 0] = (patch * zz).sum() / s
        out[i, 1] = (patch * yy).sum() / s
        out[i, 2] = (patch * xx).sum() / s
    
    return out

print("Detection module loaded")

Detection module loaded


In [4]:
def link_motion_enhanced(frames: list, 
                        max_link_um: float = 8.0,
                        motion_weight: float = 0.7,
                        max_miss: int = 2) -> TrackGraph:
    """Enhanced motion-based tracking with velocity prediction."""
    
    node_ids = []
    node_t = []
    node_z = []
    node_y = []
    node_x = []
    frame_ids = []
    nid = 1
    
    # Initialize tracks
    class Track:
        __slots__ = ['pos', 'vel', 'node_id', 'miss', 'positions', 'alive']
        def __init__(self, pos, node_id):
            self.pos = pos.copy()
            self.vel = np.zeros(3)
            self.node_id = node_id
            self.miss = 0
            self.positions = [pos.copy()]
            self.alive = True
    
    # First frame
    tracks = []
    for t, coords in enumerate(frames):
        ids_t = []
        for coord in coords:
            node_ids.append(nid)
            node_t.append(t)
            node_z.append(coord[0])
            node_y.append(coord[1])
            node_x.append(coord[2])
            ids_t.append(nid)
            if t == 0:
                tracks.append(Track(coord, nid))
            nid += 1
        frame_ids.append(ids_t)
    
    if len(frames) == 1:
        return TrackGraph(
            node_t=np.array(node_t, dtype=np.int64),
            node_z=np.array(node_z, dtype=np.float64),
            node_y=np.array(node_y, dtype=np.float64),
            node_x=np.array(node_x, dtype=np.float64),
            node_ids=np.array(node_ids, dtype=np.int64),
            edges=np.array([], dtype=np.int64).reshape(-1, 2),
            meta={}
        )
    
    edges = []
    
    for t in range(1, len(frames)):
        current_coords = frames[t]
        if len(current_coords) == 0:
            # No detections, just predict forward
            for track in tracks:
                track.miss += 1
                track.pos = track.pos + track.vel
                if track.miss > max_miss:
                    track.alive = False
            tracks = [t for t in tracks if t.alive]
            continue
        
        # Predict positions
        pred_pos = []
        for track in tracks:
            if len(track.positions) >= 2:
                vel = track.positions[-1] - track.positions[-2]
                track.vel = 0.6 * vel + 0.4 * track.vel
            pred = track.pos + track.vel * (1 + track.miss * 0.1)
            pred_pos.append(pred)
        
        # Match
        if tracks:
            cost_matrix = np.zeros((len(tracks), len(current_coords)))
            for i, pred in enumerate(pred_pos):
                max_dist = max_link_um * (1 + tracks[i].miss * 0.2)
                for j, coord in enumerate(current_coords):
                    dist = np.linalg.norm((pred - coord) * SCALE)
                    cost_matrix[i, j] = dist if dist <= max_dist else 1e6
            
            row_ind, col_ind = linear_sum_assignment(cost_matrix)
            matched_tracks = set()
            matched_dets = set()
            
            for r, c in zip(row_ind, col_ind):
                if cost_matrix[r, c] < 1e6:
                    edges.append((tracks[r].node_id, frame_ids[t][c]))
                    tracks[r].pos = current_coords[c].copy()
                    tracks[r].positions.append(current_coords[c].copy())
                    tracks[r].node_id = frame_ids[t][c]
                    tracks[r].miss = 0
                    matched_tracks.add(r)
                    matched_dets.add(c)
        
        # Update unmatched tracks
        for i, track in enumerate(tracks):
            if i not in matched_tracks:
                track.miss += 1
                track.pos = track.pos + track.vel
                if track.miss > max_miss:
                    track.alive = False
        
        # Remove dead tracks
        tracks = [t for t in tracks if t.alive]
        
        # Add new tracks for unmatched detections
        for c in range(len(current_coords)):
            if c not in matched_dets:
                new_track = Track(current_coords[c], frame_ids[t][c])
                tracks.append(new_track)
    
    # Build graph
    return TrackGraph(
        node_t=np.array(node_t, dtype=np.int64),
        node_z=np.array(node_z, dtype=np.float64),
        node_y=np.array(node_y, dtype=np.float64),
        node_x=np.array(node_x, dtype=np.float64),
        node_ids=np.array(node_ids, dtype=np.int64),
        edges=np.array(edges, dtype=np.int64).reshape(-1, 2),
        meta={}
    )

print("Tracking module loaded")

Tracking module loaded


In [5]:
def close_gaps_enhanced(frames: list, g: TrackGraph, max_gap: int = 2,
                       gap_dist_um: float = 8.0) -> TrackGraph:
    """Enhanced gap closing with interpolation."""
    if g.n_edges == 0:
        return g
    
    coords = {int(nid): (int(g.node_t[i]), g.node_z[i], g.node_y[i], g.node_x[i])
              for i, nid in enumerate(g.node_ids)}
    
    has_out = set(int(s) for s, _ in g.edges)
    has_in = set(int(t) for _, t in g.edges)
    
    ends_by_t = defaultdict(list)
    starts_by_t = defaultdict(list)
    
    for nid, (t, z, y, x) in coords.items():
        if nid not in has_out:
            ends_by_t[t].append(nid)
        if nid not in has_in:
            starts_by_t[t].append(nid)
    
    new_nodes = []
    new_edges = []
    next_id = int(g.node_ids.max()) + 1 if g.n_nodes else 1
    
    for gap in range(1, max_gap + 1):
        for t, ends in ends_by_t.items():
            starts = starts_by_t.get(t + gap + 1, [])
            if not starts:
                continue
            
            ec = np.array([[coords[e][1], coords[e][2], coords[e][3]] for e in ends]) * SCALE
            sc = np.array([[coords[s][1], coords[s][2], coords[s][3]] for s in starts]) * SCALE
            
            if len(ec) == 0 or len(sc) == 0:
                continue
            
            d = np.sqrt(((ec[:, None, :] - sc[None, :, :])**2).sum(axis=2))
            thr = gap_dist_um * (gap + 1)
            cost = np.where(d <= thr, d, 1e6)
            
            row_ind, col_ind = linear_sum_assignment(cost)
            used_s = set()
            
            for r, c in zip(row_ind, col_ind):
                if d[r, c] > thr or ends[r] in has_out or starts[c] in used_s:
                    continue
                
                e_id, s_id = ends[r], starts[c]
                te, ze, ye, xe = coords[e_id]
                ts, zs, ys, xs = coords[s_id]
                
                prev = e_id
                for k in range(1, gap + 1):
                    frac = k / (gap + 1)
                    zi = ze + (zs - ze) * frac
                    yi = ye + (ys - ye) * frac
                    xi = xe + (xs - xe) * frac
                    nid = next_id
                    next_id += 1
                    new_nodes.append((te + k, zi, yi, xi, nid))
                    new_edges.append((prev, nid))
                    prev = nid
                new_edges.append((prev, s_id))
                has_out.add(e_id)
                used_s.add(c)
    
    if not new_nodes:
        return g
    
    nt = np.concatenate([g.node_t, np.array([n[0] for n in new_nodes], dtype=np.int64)])
    nz = np.concatenate([g.node_z, np.array([n[1] for n in new_nodes])])
    ny = np.concatenate([g.node_y, np.array([n[2] for n in new_nodes])])
    nx = np.concatenate([g.node_x, np.array([n[3] for n in new_nodes])])
    nid = np.concatenate([g.node_ids, np.array([n[4] for n in new_nodes], dtype=np.int64)])
    edges = np.concatenate([g.edges, np.array(new_edges, dtype=np.int64).reshape(-1, 2)])
    
    return TrackGraph(node_t=nt, node_z=nz, node_y=ny, node_x=nx, 
                     node_ids=nid, edges=edges, meta=g.meta)

def prune_isolated(g: TrackGraph) -> TrackGraph:
    """Remove nodes not referenced by any edge."""
    if g.n_edges == 0:
        return g
    
    used = set(int(x) for x in g.edges.reshape(-1))
    keep = np.array([i for i, nid in enumerate(g.node_ids) if int(nid) in used])
    
    if len(keep) == len(g.node_ids):
        return g
    
    return TrackGraph(
        node_t=g.node_t[keep], node_z=g.node_z[keep], 
        node_y=g.node_y[keep], node_x=g.node_x[keep],
        node_ids=g.node_ids[keep], edges=g.edges, meta=g.meta
    )

def smooth_tracks(g: TrackGraph) -> TrackGraph:
    """Smooth track positions using temporal filtering."""
    if g.n_nodes < 5:
        return g
    
    # Build adjacency
    adj = defaultdict(list)
    for src, tgt in g.edges:
        adj[src].append(tgt)
        adj[tgt].append(src)
    
    # Find tracks (connected components)
    visited = set()
    tracks = []
    
    for node in g.node_ids:
        if node in visited:
            continue
        stack = [node]
        track = []
        while stack:
            curr = stack.pop()
            if curr in visited:
                continue
            visited.add(curr)
            track.append(curr)
            for neighbor in adj[curr]:
                if neighbor not in visited:
                    stack.append(neighbor)
        if len(track) >= 5:
            tracks.append(track)
    
    # Smooth each track
    for track in tracks:
        # Get positions and times
        positions = []
        times = []
        for nid in track:
            idx = np.where(g.node_ids == nid)[0][0]
            positions.append([g.node_z[idx], g.node_y[idx], g.node_x[idx]])
            times.append(g.node_t[idx])
        
        positions = np.array(positions)
        
        # Check if times are sequential
        if len(np.unique(times)) == len(times):
            # Smooth positions
            smoothed = gaussian_filter1d(positions, sigma=0.5, axis=0, mode='nearest')
            
            # Update positions if not too far
            for i, nid in enumerate(track):
                idx = np.where(g.node_ids == nid)[0][0]
                dist = np.linalg.norm((positions[i] - smoothed[i]) * SCALE)
                if dist < 3.0:
                    g.node_z[idx] = smoothed[i, 0]
                    g.node_y[idx] = smoothed[i, 1]
                    g.node_x[idx] = smoothed[i, 2]
    
    return g

print("Post-processing module loaded")

Post-processing module loaded


In [6]:
def process_dataset(zarr_path: str, 
                   xy_downsample: int = 2,
                   min_distance_um: float = 2.5,
                   rel_threshold: float = 0.025,
                   abs_percentile: float = 40.0,
                   max_peaks: int = 60000,
                   max_link_um: float = 8.0,
                   motion_weight: float = 0.7,
                   max_miss: int = 2,
                   close_gaps: bool = True,
                   max_gap: int = 2,
                   gap_dist_um: float = 8.0,
                   refine: bool = True,
                   smooth: bool = True) -> TrackGraph:
    
    # Load volume
    vol_meta = open_image(zarr_path)
    n_t = vol_meta.n_t
    
    # Detect cells in each frame
    frames = []
    for t in range(n_t):
        vol = vol_meta.frame(t)
        coords = detect_blobs_enhanced(
            vol,
            xy_downsample=xy_downsample,
            min_distance_um=min_distance_um,
            rel_threshold=rel_threshold,
            abs_percentile=abs_percentile,
            max_peaks=max_peaks
        )
        
        if refine and len(coords) > 0:
            coords = refine_centroids(vol, coords)
        
        frames.append(coords)
        
        # Free memory
        del vol
        if t % 10 == 0:
            gc.collect()
    
    # Link frames
    g = link_motion_enhanced(
        frames,
        max_link_um=max_link_um,
        motion_weight=motion_weight,
        max_miss=max_miss
    )
    
    # Post-process
    if close_gaps:
        g = close_gaps_enhanced(frames, g, max_gap=max_gap, gap_dist_um=gap_dist_um)
    
    g = prune_isolated(g)
    
    if smooth:
        g = smooth_tracks(g)
    
    return g

print("Pipeline loaded")

Pipeline loaded


In [7]:
def graph_to_rows(name: str, g: TrackGraph) -> list:
    """Convert graph to submission rows."""
    rows = []
    
    # Node rows
    for i in range(g.n_nodes):
        rows.append({
            "dataset": name,
            "row_type": "node",
            "node_id": int(g.node_ids[i]),
            "t": int(g.node_t[i]),
            "z": int(round(g.node_z[i])),
            "y": int(round(g.node_y[i])),
            "x": int(round(g.node_x[i])),
            "source_id": -1,
            "target_id": -1,
        })
    
    # Edge rows
    for src, tgt in g.edges:
        rows.append({
            "dataset": name,
            "row_type": "edge",
            "node_id": -1,
            "t": -1,
            "z": -1,
            "y": -1,
            "x": -1,
            "source_id": int(src),
            "target_id": int(tgt),
        })
    
    return rows

def create_submission(results: dict, output_path: str) -> pd.DataFrame:
    """Create submission file."""
    all_rows = []
    for name, g in results.items():
        all_rows.extend(graph_to_rows(name, g))
    
    df = pd.DataFrame(all_rows, columns=[
        "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"
    ])
    df.index.name = "id"
    df.to_csv(output_path)
    return df

print("Submission generator loaded")

Submission generator loaded


In [8]:
# Configuration optimized for competition
CONFIG = {
    'xy_downsample': 2,
    'min_distance_um': 2.5,
    'rel_threshold': 0.03,
    'abs_percentile': 40.0,
    'max_peaks': 60000,
    'max_link_um': 8.0,
    'motion_weight': 0.7,
    'max_miss': 2,
    'close_gaps': True,
    'max_gap': 2,
    'gap_dist_um': 8.0,
    'refine': True,
    'smooth': True,
}

# Find test directory
def find_test_dir():
    # Check common locations
    candidates = [
        "/kaggle/input/biohub-cell-tracking-during-development/test",
        "/kaggle/input/competitions/biohub-cell-tracking-during-development/test",
        os.environ.get("TEST_DIR", ""),
    ]
    
    for c in candidates:
        if c and os.path.isdir(c):
            return c
    
    # Search
    for root, dirs, files in os.walk("/kaggle/input"):
        if os.path.basename(root) == "test":
            zarrs = [d for d in dirs if d.endswith(".zarr")]
            if zarrs:
                return root
    
    raise FileNotFoundError("Test directory not found")

print("Configuration ready")

Configuration ready


In [9]:
def main():
    print("Starting Biohub Cell Tracking Pipeline...")
    start_time = time.time()
    
    # Find test directory
    test_dir = find_test_dir()
    print(f"Test directory: {test_dir}")
    
    # Get datasets
    datasets = sorted([d[:-5] for d in os.listdir(test_dir) if d.endswith(".zarr")])
    print(f"Found {len(datasets)} datasets")
    
    # Process each dataset
    results = {}
    for i, name in enumerate(datasets):
        print(f"\nProcessing {i+1}/{len(datasets)}: {name}")
        t0 = time.time()
        
        zarr_path = os.path.join(test_dir, name + ".zarr")
        
        try:
            g = process_dataset(zarr_path, **CONFIG)
            results[name] = g
            print(f"  Nodes: {g.n_nodes}, Edges: {g.n_edges}")
            print(f"  Time: {time.time() - t0:.1f}s")
        except Exception as e:
            print(f"  ERROR processing {name}: {e}")
            # Create empty graph for failed dataset
            results[name] = TrackGraph(
                node_t=np.array([], dtype=np.int64),
                node_z=np.array([], dtype=np.float64),
                node_y=np.array([], dtype=np.float64),
                node_x=np.array([], dtype=np.float64),
                node_ids=np.array([], dtype=np.int64),
                edges=np.array([], dtype=np.int64).reshape(-1, 2),
                meta={}
            )
        
        # Free memory
        gc.collect()
    
    # Create submission
    print("\nCreating submission...")
    df = create_submission(results, "submission.csv")
    
    total_time = time.time() - start_time
    print(f"\nDone! Total time: {total_time:.1f}s")
    print(f"Submission rows: {len(df)}")
    print(f"Output file: submission.csv")
    
    # Show sample
    print("\nSample submission:")
    print(df.head(10))

# Run the pipeline
if __name__ == "__main__":
    main()

Starting Biohub Cell Tracking Pipeline...
Test directory: /kaggle/input/competitions/biohub-cell-tracking-during-development/test
Found 4 datasets

Processing 1/4: 44b6_0113de3b
  Nodes: 28897, Edges: 27806
  Time: 151.7s

Processing 2/4: 44b6_0b24845f
  Nodes: 61239, Edges: 59335
  Time: 308.2s

Processing 3/4: 6bba_05b6850b
  Nodes: 8825, Edges: 8361
  Time: 107.2s

Processing 4/4: 6bba_05db0fb1
  Nodes: 81689, Edges: 79479
  Time: 461.2s

Creating submission...

Done! Total time: 1031.9s
Submission rows: 355631
Output file: submission.csv

Sample submission:
          dataset row_type  node_id  t   z    y    x  source_id  target_id
id                                                                        
0   44b6_0113de3b     node        1  0  41    6  172         -1         -1
1   44b6_0113de3b     node        2  0  20  179   82         -1         -1
2   44b6_0113de3b     node        3  0  17  104   84         -1         -1
3   44b6_0113de3b     node        4  0  13  142   72     

In [10]:
# Quick validation of submission format
def validate_submission(submission_path="submission.csv"):
    df = pd.read_csv(submission_path)
    
    print(f"Total rows: {len(df)}")
    print(f"Datasets: {df['dataset'].nunique()}")
    print(f"Node rows: {len(df[df['row_type'] == 'node'])}")
    print(f"Edge rows: {len(df[df['row_type'] == 'edge'])}")
    
    # Check required columns
    required_cols = ['id', 'dataset', 'row_type', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']
    missing_cols = set(required_cols) - set(df.columns)
    if missing_cols:
        print(f"WARNING: Missing columns: {missing_cols}")
    else:
        print("All required columns present")
    
    # Check format
    nodes = df[df['row_type'] == 'node']
    edges = df[df['row_type'] == 'edge']
    
    if len(nodes) > 0:
        print(f"Node ID range: {nodes['node_id'].min()} - {nodes['node_id'].max()}")
    
    if len(edges) > 0:
        print(f"Edge sources: {edges['source_id'].min()} - {edges['source_id'].max()}")
        print(f"Edge targets: {edges['target_id'].min()} - {edges['target_id'].max()}")
        
    return df

# Validate after running main
# Uncomment to validate:
#df = validate_submission("submission.csv")

In [11]:
# Test on first dataset only (useful for debugging)
def test_single():
    test_dir = find_test_dir()
    datasets = [d[:-5] for d in os.listdir(test_dir) if d.endswith(".zarr")]
    
    if not datasets:
        print("No datasets found")
        return
    
    # Test on first dataset
    name = datasets[0]
    print(f"Testing on: {name}")
    
    zarr_path = os.path.join(test_dir, name + ".zarr")
    g = process_dataset(zarr_path, **CONFIG)
    
    print(f"Nodes: {g.n_nodes}, Edges: {g.n_edges}")
    
    # Create submission for this single dataset
    result = {name: g}
    df = create_submission(result, "test_submission.csv")
    print(f"Created test_submission.csv with {len(df)} rows")
    
    return g

# Uncomment to test:
#test_single()